# Whale Shark Dataset Starter — Veritas AI Fellowship

**Project:** Machine Learning Prediction of Whale Shark Presence to Reduce Tour-Boat Crowding at Mafia Island, Tanzania
**Author:** Patrick Silingardi · **Mentor:** Samuel Haghshenas (University of Oxford)
**Rebuilt:** 25 August 2026

---

## What this notebook does

It builds the **Plan B (global) working dataset** from open sources, end to end, with no
permissions required. Run it top to bottom in Colab and it produces:

| Output | What it is |
|---|---|
| `whaleshark_global_clean.csv` | analysis-ready global dataset — the Plan B base |
| `whaleshark_obis_raw.csv` | raw OBIS pull (audit trail) |
| `whaleshark_gbif_raw.csv` | raw GBIF pull (audit trail) |
| `qc_report.txt` | what was dropped and why |

## What it deliberately does NOT do

It does not model anything. Building a model before the seasonal climatology baseline exists
is the fastest way to produce an impressive number that means nothing — see §12.

## Reliability corrections built in

These answer the mentor's question *"how reliable are the data?"* directly:

1. **Telemetry is separated and thinned.** ~80% of GBIF records are `MACHINE_OBSERVATION`
   satellite pings from a few dozen tagged animals — pseudo-replicates, not independent
   observations. Thinned to max 1 position per dataset/day/0.1° cell.
2. **GBIF and OBIS are deduplicated.** They republish the same underlying datasets. Treating
   them as two independent sources double-counts.
3. **Coordinate obfuscation is flagged.** iNaturalist obscures threatened-species records to
   ~31 km. For a bay-scale study that error is the size of the study area.
4. **Effort bias is quantified**, not assumed away. Record density measures researcher and
   tourist density, not shark density.
5. **Country codes are normalised** to ISO-2 (OBIS returns names, GBIF returns codes).


In [ ]:
# ---- Setup -------------------------------------------------------------
# Colab has requests + pandas already. ephem is for lunar phase.
!pip install -q ephem

import json, time, math, os, sys
from datetime import datetime, timezone

import requests
import pandas as pd
import numpy as np

pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 50)

print("pandas", pd.__version__)
print("ready")

## 1. Configuration

`MAFIA_BBOX` is the Plan A study area: Kilindoni Bay and the waters around Mafia Island.
It is used in §11 to show empirically how little open data sits inside it.

In [ ]:
# ---- Configuration -----------------------------------------------------
SPECIES = "Rhincodon typus"

# Plan A study area (lon_min, lon_max, lat_min, lat_max) — Mafia Island, Tanzania
MAFIA_BBOX = dict(lon_min=39.2, lon_max=39.9, lat_min=-8.3, lat_max=-7.5)

# Telemetry thinning resolution (degrees). 0.1 deg ~ 11 km.
THIN_CELL_DEG = 0.1

# Coordinate uncertainty above which a record is unusable for fine spatial work (metres)
MAX_UNCERTAINTY_M = 50_000

OUT_CLEAN = "whaleshark_global_clean.csv"
OUT_OBIS  = "whaleshark_obis_raw.csv"
OUT_GBIF  = "whaleshark_gbif_raw.csv"
OUT_QC    = "qc_report.txt"

QC = []   # collects lines for the QC report

def qc(msg):
    """Log a line to both stdout and the QC report."""
    print(msg)
    QC.append(msg)

qc(f"Run started: {datetime.now(timezone.utc).isoformat()}")
qc(f"Species: {SPECIES}")

## 2. OBIS pull

OBIS is the preferred aggregator for this project because it ships environmental covariates
**already joined to each record** — `sst`, `sss`, `bathymetry`, `shoredistance`. That is a
complete first feature table with zero raster engineering.

Deep paging uses the `after` parameter (cursor on record id), not offset.

In [ ]:
# ---- OBIS pull ---------------------------------------------------------
OBIS_URL = "https://api.obis.org/v3/occurrence"

def fetch_obis(species=SPECIES, page_size=5000, max_pages=50, pause=0.5):
    """Page through the OBIS occurrence endpoint using the `after` cursor."""
    rows, after, pages = [], None, 0
    while pages < max_pages:
        params = {"scientificname": species, "size": page_size}
        if after:
            params["after"] = after
        for attempt in range(3):
            try:
                r = requests.get(OBIS_URL, params=params, timeout=120)
                r.raise_for_status()
                break
            except Exception as e:
                if attempt == 2:
                    raise
                print(f"  retry {attempt+1} after error: {e}")
                time.sleep(3 * (attempt + 1))
        payload = r.json()
        batch = payload.get("results", [])
        if not batch:
            break
        rows.extend(batch)
        after = batch[-1].get("id")
        pages += 1
        print(f"  OBIS page {pages}: +{len(batch)} (total {len(rows)})")
        if len(batch) < page_size or not after:
            break
        time.sleep(pause)
    return pd.DataFrame(rows)

obis_raw = fetch_obis()
qc(f"OBIS raw records pulled: {len(obis_raw)}")
obis_raw.to_csv(OUT_OBIS, index=False)
obis_raw.head(3)

## 3. GBIF pull

The taxon key is resolved dynamically rather than hard-coded, so the notebook does not silently
break if GBIF re-keys the species.

GBIF caps a page at 300 records and holds roughly 16,000 for this species, so a serial loop
means ~55 round trips one after another — slow enough to look hung. These run **8 at a time**,
and the cell asks GBIF for the total first so it can print real progress and an ETA.

If you want it faster still, set `GBIF_INCLUDE_TELEMETRY = False`: that drops to ~2,500 records
(about 10x quicker). Section 6 thins the telemetry to almost nothing anyway — the only thing you
lose is the full-record effort-bias breakdown in section 9, so leave it True the first time.


In [ ]:
# ---- GBIF pull (parallel) ---------------------------------------------
# GBIF caps a page at 300 records and holds ~16k for this species, so a serial
# loop is ~55 round trips one after another. These run 8 at a time instead.
from concurrent.futures import ThreadPoolExecutor, as_completed

GBIF_MATCH = "https://api.gbif.org/v1/species/match"
GBIF_OCC   = "https://api.gbif.org/v1/occurrence/search"

# Set to False to skip satellite telemetry entirely: ~2.5k records instead of ~16k,
# roughly 10x faster. Section 6 thins telemetry to near-nothing anyway, so this
# costs very little signal. Leave True the first time so the effort-bias numbers
# in section 9 are computed on the full record.
GBIF_INCLUDE_TELEMETRY = True
GBIF_WORKERS = 8
GBIF_PAGE = 300

HUMAN_BASES = ["HUMAN_OBSERVATION", "OBSERVATION", "PRESERVED_SPECIMEN",
               "MATERIAL_SAMPLE", "OCCURRENCE"]

def gbif_taxon_key(species=SPECIES):
    r = requests.get(GBIF_MATCH, params={"name": species}, timeout=60)
    r.raise_for_status()
    m = r.json()
    print(f"  GBIF taxon match: {m.get('scientificName')} (key={m.get('usageKey')})")
    return m.get("usageKey")

def _base_params(key):
    p = {"taxonKey": key, "hasCoordinate": "true"}
    if not GBIF_INCLUDE_TELEMETRY:
        p["basisOfRecord"] = HUMAN_BASES
    return p

def _gbif_page(key, offset):
    """Fetch one page. Returns [] rather than raising, so one bad page can't kill the run."""
    params = {**_base_params(key), "limit": GBIF_PAGE, "offset": offset}
    for attempt in range(4):
        try:
            r = requests.get(GBIF_OCC, params=params, timeout=120)
            r.raise_for_status()
            return r.json().get("results", [])
        except Exception as e:
            if attempt == 3:
                print(f"  ! page at offset {offset} failed after 4 tries: {e}")
                return []
            time.sleep(2 * (attempt + 1))

def fetch_gbif(species=SPECIES):
    key = gbif_taxon_key(species)

    # Ask for the total first, so we know how many pages to request and can show progress
    r = requests.get(GBIF_OCC, params={**_base_params(key), "limit": 0}, timeout=60)
    r.raise_for_status()
    total = min(r.json().get("count", 0), 100_000)   # GBIF caps paging at offset 100k
    offsets = list(range(0, total, GBIF_PAGE))
    print(f"  {total} records to fetch in {len(offsets)} pages, {GBIF_WORKERS} at a time")

    rows, done = [], 0
    t0 = time.time()
    with ThreadPoolExecutor(max_workers=GBIF_WORKERS) as pool:
        futures = {pool.submit(_gbif_page, key, off): off for off in offsets}
        for fut in as_completed(futures):
            rows.extend(fut.result())
            done += 1
            if done % 10 == 0 or done == len(offsets):
                el = time.time() - t0
                eta = el / done * (len(offsets) - done)
                print(f"  {done}/{len(offsets)} pages · {len(rows)} records · "
                      f"{el:.0f}s elapsed, ~{eta:.0f}s left")
    return pd.DataFrame(rows)

gbif_raw = fetch_gbif()
qc(f"GBIF raw records pulled: {len(gbif_raw)}")
gbif_raw.to_csv(OUT_GBIF, index=False)
gbif_raw.head(3)


## 4. Normalisation to a common schema

Both sources collapse into one table with the same columns. The schema is deliberately
**source-agnostic** — `lat, lon, datetime, source, basis, dataset_id` — so that an MMF or
Sharkbook extract arriving later can be appended without rewriting anything downstream.

**Date parsing deserves a warning.** Both aggregators mix formats inside the same column:
`2005-05-24T12:00:00Z`, `2022-04-22 22:21:59`, `2016-12-02`, `2015-01-12T09:06:00-06:00`.
pandas 2.x infers a single format from the first value and quietly coerces everything else to
`NaT`. On this dataset that silently destroyed about 7,000 valid OBIS records — a 42% loss that
looked like a property of the data rather than a bug. `format="mixed"` parses each value on its
own terms, and OBIS's `date_mid` (epoch milliseconds) catches the stragglers. The cell prints
the parse rate per source so this can never fail silently again.

`dataset_id` is kept on every row on purpose: parts of the underlying data are **CC BY-NC**
(some iNaturalist, Diveboard, Observation.org, OBIS-SEAMAP, the ECOCEAN photo-ID library). Fine
for a capstone, a blocker if the product were ever sold to operators. Row-level provenance is
what makes that checkable later.


In [ ]:
# ---- Country normalisation --------------------------------------------
# OBIS returns country names, GBIF returns ISO-2 codes. Without this, the analysis
# in section 9 splits "Australia" and "AU" into two different countries.
NAME_TO_ISO2 = {
    "australia": "AU", "united states": "US", "united states of america": "US",
    "mexico": "MX", "philippines": "PH", "indonesia": "ID", "maldives": "MV",
    "thailand": "TH", "tanzania": "TZ", "united republic of tanzania": "TZ",
    "mozambique": "MZ", "south africa": "ZA", "india": "IN", "sri lanka": "LK",
    "japan": "JP", "taiwan": "TW", "china": "CN", "malaysia": "MY", "belize": "BZ",
    "honduras": "HN", "ecuador": "EC", "brazil": "BR", "seychelles": "SC",
    "madagascar": "MG", "kenya": "KE", "djibouti": "DJ", "saudi arabia": "SA",
    "qatar": "QA", "oman": "OM", "united arab emirates": "AE", "egypt": "EG",
    "papua new guinea": "PG", "new caledonia": "NC", "france": "FR", "spain": "ES",
    "portugal": "PT", "cuba": "CU", "panama": "PA", "costa rica": "CR",
    "colombia": "CO", "peru": "PE", "venezuela": "VE", "fiji": "FJ",
    "viet nam": "VN", "vietnam": "VN", "cabo verde": "CV", "cape verde": "CV",
}

def to_iso2(v):
    if v is None or (isinstance(v, float) and pd.isna(v)):
        return None
    s = str(v).strip()
    if len(s) == 2 and s.isalpha():
        return s.upper()
    return NAME_TO_ISO2.get(s.lower(), s[:2].upper() if s else None)

def pick(df, col, default=None):
    """Return a column if it exists, otherwise a column of the default value."""
    return df[col] if col in df.columns else pd.Series([default] * len(df), index=df.index)

# ---- DATE PARSING --------------------------------------------------------
# Both aggregators mix date formats in the same column: '2005-05-24T12:00:00Z',
# '2022-04-22 22:21:59', '2016-12-02', '2015-01-12T09:06:00-06:00'. pandas 2.x infers
# ONE format from the first value and coerces everything else to NaT, so a plain
# to_datetime(..., utc=True) silently discarded ~7,000 perfectly good OBIS records.
# format="mixed" parses each value on its own terms. OBIS also ships date_mid as
# epoch milliseconds, which is used as a fallback for the stragglers.
def parse_dates(series, fallback_ms=None):
    dt = pd.to_datetime(series, errors="coerce", utc=True, format="mixed")
    if fallback_ms is not None:
        fb = pd.to_datetime(fallback_ms, unit="ms", errors="coerce", utc=True)
        dt = dt.fillna(fb)
    return dt

# ---- OBIS -> common schema ----
obis = pd.DataFrame({
    "record_id":   pick(obis_raw, "id"),
    "source":      "OBIS",
    "lat":         pd.to_numeric(pick(obis_raw, "decimalLatitude"), errors="coerce"),
    "lon":         pd.to_numeric(pick(obis_raw, "decimalLongitude"), errors="coerce"),
    "basis":       pick(obis_raw, "basisOfRecord"),
    "uncertainty_m": pd.to_numeric(pick(obis_raw, "coordinateUncertaintyInMeters"), errors="coerce"),
    "country_raw": pick(obis_raw, "country"),
    "dataset_id":  pick(obis_raw, "dataset_id"),
    "individual_count": pd.to_numeric(pick(obis_raw, "individualCount"), errors="coerce"),
    # OBIS ships these joined already — free covariates
    "sst":         pd.to_numeric(pick(obis_raw, "sst"), errors="coerce"),
    "sss":         pd.to_numeric(pick(obis_raw, "sss"), errors="coerce"),
    "bathymetry":  pd.to_numeric(pick(obis_raw, "bathymetry"), errors="coerce"),
    "shoredistance": pd.to_numeric(pick(obis_raw, "shoredistance"), errors="coerce"),
})
obis["datetime"] = parse_dates(pick(obis_raw, "eventDate"),
                               pd.to_numeric(pick(obis_raw, "date_mid"), errors="coerce"))

# ---- GBIF -> common schema ----
gbif = pd.DataFrame({
    "record_id":   pick(gbif_raw, "key").astype(str),
    "source":      "GBIF",
    "lat":         pd.to_numeric(pick(gbif_raw, "decimalLatitude"), errors="coerce"),
    "lon":         pd.to_numeric(pick(gbif_raw, "decimalLongitude"), errors="coerce"),
    "basis":       pick(gbif_raw, "basisOfRecord"),
    "uncertainty_m": pd.to_numeric(pick(gbif_raw, "coordinateUncertaintyInMeters"), errors="coerce"),
    "country_raw": pick(gbif_raw, "countryCode"),
    "dataset_id":  pick(gbif_raw, "datasetKey"),
    "individual_count": pd.to_numeric(pick(gbif_raw, "individualCount"), errors="coerce"),
    "sst": np.nan, "sss": np.nan, "bathymetry": np.nan, "shoredistance": np.nan,
})
gbif["datetime"] = parse_dates(pick(gbif_raw, "eventDate"))

# Report the parse rate per source: a silent drop here is the most damaging
# failure mode in the whole pipeline, so it gets its own line in the QC report.
for nm, d in (("OBIS", obis), ("GBIF", gbif)):
    ok, tot = int(d["datetime"].notna().sum()), len(d)
    qc(f"{nm} dates parsed: {ok}/{tot} ({ok/max(tot,1):.1%})")

df = pd.concat([obis, gbif], ignore_index=True)
df["country"] = df["country_raw"].map(to_iso2)
df["basis"] = df["basis"].astype(str).str.upper().str.replace(" ", "_")

qc(f"Combined before cleaning: {len(df)} rows ({len(obis)} OBIS + {len(gbif)} GBIF)")
df["basis"].value_counts().head(10)


## 5. Deduplication — GBIF ↔ OBIS

This step matters more than it looks. GBIF and OBIS are **not two independent sources**: they
republish the same underlying datasets. Counting a record twice inflates every downstream
statistic and makes a well-sampled site look twice as well-sampled.

Two passes: exact `record_id`, then the practical key (rounded coordinates, date, basis).

In [ ]:
# ---- Deduplication -----------------------------------------------------
before = len(df)

# Pass 1: identical record ids across sources
df = df.drop_duplicates(subset=["record_id"], keep="first")
after_id = len(df)

# Pass 2: same place, same day, same kind of record -> almost certainly the same observation
df["_lat5"] = df["lat"].round(5)
df["_lon5"] = df["lon"].round(5)
df["_day"]  = df["datetime"].dt.date

# Keep OBIS preferentially: it carries the free environmental covariates
df["_src_rank"] = (df["source"] == "OBIS").map({True: 0, False: 1})
df = (df.sort_values("_src_rank")
        .drop_duplicates(subset=["_lat5", "_lon5", "_day", "basis"], keep="first"))

df = df.drop(columns=["_lat5", "_lon5", "_day", "_src_rank"])

qc(f"Dedup: {before} -> {after_id} (by record_id) -> {len(df)} (by lat/lon/day/basis)")
qc(f"  removed as duplicates: {before - len(df)}")
df["source"].value_counts()

## 6. Telemetry: separate, then thin

This is the single most important reliability correction in the notebook.

Roughly 80% of GBIF records for this species are `MACHINE_OBSERVATION` — Argos satellite pings
from a few dozen tagged animals. One shark transmitting for six months can generate thousands of
records. Left untouched, a model trained on this learns *where the tagged animals were*, not
where whale sharks are, and the effective sample size is a few dozen rather than tens of
thousands.

Thinning to **one position per dataset / day / 0.1° cell** keeps the ecological signal while
removing the pseudo-replication.

In [ ]:
# ---- Telemetry separation and thinning ---------------------------------
TELEMETRY_BASES = {"MACHINE_OBSERVATION"}

df["is_telemetry"] = df["basis"].isin(TELEMETRY_BASES)

n_tel = int(df["is_telemetry"].sum())
n_hum = int((~df["is_telemetry"]).sum())
qc(f"Telemetry records: {n_tel} ({n_tel / max(len(df),1):.1%})")
qc(f"Human observations: {n_hum} ({n_hum / max(len(df),1):.1%})")

tel = df[df["is_telemetry"]].copy()
hum = df[~df["is_telemetry"]].copy()

if len(tel):
    tel["_cell_lat"] = (tel["lat"] / THIN_CELL_DEG).round()
    tel["_cell_lon"] = (tel["lon"] / THIN_CELL_DEG).round()
    tel["_day"] = tel["datetime"].dt.date
    before_thin = len(tel)
    tel = tel.drop_duplicates(subset=["dataset_id", "_day", "_cell_lat", "_cell_lon"], keep="first")
    tel = tel.drop(columns=["_cell_lat", "_cell_lon", "_day"])
    qc(f"Telemetry thinned ({THIN_CELL_DEG} deg / day / dataset): {before_thin} -> {len(tel)}")

df = pd.concat([hum, tel], ignore_index=True)
qc(f"After thinning, total rows: {len(df)}")

## 7. Quality flags and cleaning

Coordinate obfuscation is flagged rather than dropped, because it is only a problem at fine
spatial scales. A record fuzzed to 31 km is useless for within-bay work and perfectly fine for a
global site-level model — so the decision is pushed downstream to whoever is modelling, instead
of being silently baked in here.

In [ ]:
# ---- Quality flags and cleaning ----------------------------------------
before = len(df)

# Valid coordinates
bad_coords = df["lat"].isna() | df["lon"].isna() | \
             (df["lat"].abs() > 90) | (df["lon"].abs() > 180)
qc(f"Dropped for invalid/missing coordinates: {int(bad_coords.sum())}")
df = df[~bad_coords].copy()

# Null island
null_island = (df["lat"].abs() < 0.001) & (df["lon"].abs() < 0.001)
qc(f"Dropped as (0,0) null island: {int(null_island.sum())}")
df = df[~null_island].copy()

# Usable date
no_date = df["datetime"].isna()
qc(f"Dropped for missing/unparseable date: {int(no_date.sum())}")
df = df[~no_date].copy()

# Pre-satellite era: environmental covariates cannot be reconstructed
df["year"] = df["datetime"].dt.year
too_old = df["year"] < 1981
qc(f"Dropped as pre-1981 (no satellite covariates available): {int(too_old.sum())}")
df = df[~too_old].copy()

# Obfuscation flag — NOT dropped, flagged
df["is_obscured"] = df["uncertainty_m"].fillna(0) >= 20_000
df["fine_scale_ok"] = df["uncertainty_m"].fillna(0) <= MAX_UNCERTAINTY_M

qc(f"Flagged as coordinate-obscured (>=20 km): {int(df['is_obscured'].sum())}")
qc(f"Usable for fine spatial work (<= {MAX_UNCERTAINTY_M/1000:.0f} km): {int(df['fine_scale_ok'].sum())}")
qc(f"Cleaning: {before} -> {len(df)} rows")

## 8. Temporal features and lunar phase

`doy_sin` / `doy_cos` encode day-of-year cyclically, so that 31 December and 1 January are
adjacent rather than 364 days apart — which is what a raw day number would tell the model.

Lunar phase is computed, not downloaded: no dataset is needed. `ephem` is used where available,
with a mean-synodic fallback accurate to about ±0.5 days.

In [ ]:
# ---- Temporal features -------------------------------------------------
df["month"] = df["datetime"].dt.month
df["day_of_year"] = df["datetime"].dt.dayofyear

df["doy_sin"] = np.sin(2 * np.pi * df["day_of_year"] / 365.25)
df["doy_cos"] = np.cos(2 * np.pi * df["day_of_year"] / 365.25)

# ---- Lunar phase (0 = new moon, 0.5 = full moon) ----
def moon_phase_series(dts):
    try:
        import ephem
        def one(d):
            if pd.isna(d):
                return np.nan
            prev_new = ephem.previous_new_moon(d.to_pydatetime().replace(tzinfo=None))
            next_new = ephem.next_new_moon(d.to_pydatetime().replace(tzinfo=None))
            cycle = float(next_new) - float(prev_new)
            return (float(ephem.Date(d.to_pydatetime().replace(tzinfo=None))) - float(prev_new)) / cycle
        print("  lunar phase via ephem")
        return dts.map(one)
    except Exception as e:
        print(f"  ephem unavailable ({e}) — using mean synodic approximation (+/- ~0.5 d)")
        SYNODIC = 29.530588853
        REF = pd.Timestamp("2000-01-06 18:14", tz="UTC")   # known new moon
        return ((dts - REF).dt.total_seconds() / 86400.0 % SYNODIC) / SYNODIC

df["moon_phase"] = moon_phase_series(df["datetime"])

qc(f"Temporal features built. Month coverage: {sorted(df['month'].dropna().unique().tolist())}")
df[["datetime", "month", "day_of_year", "doy_sin", "doy_cos", "moon_phase"]].head()

## 9. Effort bias — quantified, not assumed away

**This is the section to show Samuel.** It is the honest answer to "how reliable are the data?"

Record density measures where people with cameras, boats and internet access are. It does not
measure where sharks are.

Two views, because a country table on its own lies by omission: OBIS populates `country` on
under 20% of its records, so grouping by country silently describes a minority of the data and
makes Tanzania look like it has one record. The second view assigns each record to the nearest
known aggregation site **from its coordinates**, which every record has — that is the honest
comparison, and it is also the exact table Plan B is built on.


In [ ]:
# ---- Effort bias analysis ----------------------------------------------
# This cell reads columns that earlier cells create. Run 5-8 first: section 6 adds
# `is_telemetry` and thins the telemetry, section 7 drops invalid coordinates and
# dates. Running this on a freshly rebuilt `df` would produce a table that looks
# right and describes the wrong stage of the pipeline.
_missing = [c for c in ("is_telemetry", "is_obscured", "year") if c not in df.columns]
if _missing:
    raise RuntimeError(
        f"Cannot run section 9 yet: missing {_missing}. "
        "Re-run sections 5, 6, 7 and 8 in order first (section 4 rebuilds `df` from "
        "scratch, which clears everything the later cells added)."
    )

# Two views, because the country field alone is misleading: OBIS populates `country`
# on under 20% of its records, so a country table silently describes a minority of
# the data. The site view below is computed from coordinates, which every record has.
qc("")
qc("=== EFFORT BIAS ===")

no_country = int(df["country"].isna().sum())
qc(f"Records with NO country field: {no_country} of {len(df)} ({no_country/len(df):.1%})")
qc("  -> the country table below describes only the remainder. Do not quote it as")
qc("     'share of the global record' without this caveat.")
qc("")

by_country = (df.dropna(subset=["country"]).groupby("country")
                .agg(records=("record_id", "size"), telemetry=("is_telemetry", "sum"))
                .sort_values("records", ascending=False))
known = by_country["records"].sum()
by_country["share_of_known_%"] = (100 * by_country["records"] / max(known, 1)).round(1)
by_country["telemetry_%"] = (100 * by_country["telemetry"] / by_country["records"]).round(1)

qc(f"Top countries (of the {known} records that state one):")
for c, row in by_country.head(12).iterrows():
    qc(f"  {str(c):<6} {int(row['records']):>7}  ({row['share_of_known_%']:>5}% of stated, "
       f"{row['telemetry_%']:>5}% telemetry)")
qc("")

# ---- Known aggregation sites: the view that actually matters for Plan B ----
SITES = {
    "Ningaloo, AU":        (-22.00, 113.90), "Isla Mujeres, MX":    ( 21.40, -86.50),
    "Bahia de los Angeles":( 28.95,-113.56), "La Paz, MX":          ( 24.20,-110.30),
    "Donsol, PH":          ( 12.90, 123.60), "Oslob, PH":           (  9.50, 123.40),
    "Cenderawasih, ID":    ( -2.40, 134.80), "Saleh Bay, ID":       ( -8.50, 117.60),
    "South Ari, MV":       (  3.50,  72.80), "MAFIA ISLAND, TZ":    ( -7.90,  39.60),
    "Tofo, MZ":            (-23.85,  35.54), "Djibouti":            ( 11.70,  42.80),
    "Al Shaheen, QA":      ( 26.00,  52.00), "Mahe, SC":            ( -4.70,  55.50),
    "Utila, HN":           ( 16.10, -86.90), "Darwin, Galapagos":   (  1.68, -92.00),
    "N Gulf of Mexico":    ( 26.50, -85.00), "Christmas Island":    (-10.50, 105.60),
    "Taiwan":              ( 23.00, 121.50), "Nosy Be, MG":         (-13.30,  48.30),
}
RADIUS_KM = 200

def haversine_km(lat1, lon1, lat2, lon2):
    r1, r2 = np.radians(lat1), np.radians(lat2)
    dlat, dlon = np.radians(lat2 - lat1), np.radians(lon2 - lon1)
    a = np.sin(dlat / 2) ** 2 + np.cos(r1) * np.cos(r2) * np.sin(dlon / 2) ** 2
    return 6371.0 * 2 * np.arcsin(np.sqrt(a))

names = list(SITES)
dists = np.column_stack([haversine_km(df["lat"].values, df["lon"].values, la, lo)
                         for la, lo in SITES.values()])
nearest = dists.argmin(axis=1)
mind = dists.min(axis=1)
df["site"] = np.where(mind <= RADIUS_KM, np.array(names)[nearest], None)
df["site_km"] = mind

by_site = (df.dropna(subset=["site"]).groupby("site")
             .agg(records=("record_id", "size"), telemetry=("is_telemetry", "sum"))
             .sort_values("records", ascending=False))
by_site["telemetry_%"] = (100 * by_site["telemetry"] / by_site["records"]).round(1)

qc(f"Records within {RADIUS_KM} km of a known aggregation site "
   f"(computed from coordinates, so no missing-field problem):")
for s, row in by_site.iterrows():
    star = "  <<<" if "MAFIA" in s else ""
    qc(f"  {s:<24} {int(row['records']):>6}  ({row['telemetry_%']:>5}% telemetry){star}")
unassigned = int(df["site"].isna().sum())
qc(f"  {'(not near any listed site)':<24} {unassigned:>6}")
qc("")

mafia = df[df["site"] == "MAFIA ISLAND, TZ"]
qc(f"Mafia Island: {len(mafia)} records in the open global aggregators —")
qc(f"  against ~1,185 in the Sharkbook Tanzania export. The open record is not just")
qc(f"  thin, it is roughly {1185/max(len(mafia),1):.0f}x thinner than the restricted one.")
qc(f"  Record density here measures who is looking and uploading, not shark abundance.")
qc("")
qc(f"Human observations: {int((~df['is_telemetry']).sum())}")
qc(f"Telemetry (thinned): {int(df['is_telemetry'].sum())}")

by_site


## 10. Export

`whaleshark_global_clean.csv` is the Plan B base. In Colab, download the files from the folder
icon in the left sidebar, or run the last cell of this section.

In [ ]:
# ---- Export ------------------------------------------------------------
cols = ["record_id", "source", "dataset_id", "lat", "lon", "datetime", "year", "month",
        "day_of_year", "doy_sin", "doy_cos", "moon_phase", "basis", "is_telemetry",
        "uncertainty_m", "is_obscured", "fine_scale_ok", "country",
        "individual_count", "sst", "sss", "bathymetry", "shoredistance",
        "site", "site_km"]
cols = [c for c in cols if c in df.columns]

clean = df[cols].sort_values("datetime").reset_index(drop=True)
clean.to_csv(OUT_CLEAN, index=False)

qc("")
qc("=== EXPORT ===")
qc(f"{OUT_CLEAN}: {len(clean)} rows x {len(cols)} columns")
qc(f"  date range: {clean['datetime'].min()} -> {clean['datetime'].max()}")
qc(f"  with OBIS covariates (sst present): {int(clean['sst'].notna().sum())}")
qc(f"Run finished: {datetime.now(timezone.utc).isoformat()}")

with open(OUT_QC, "w") as f:
    f.write("\n".join(QC))

print()
print(f"Written: {OUT_CLEAN}, {OUT_OBIS}, {OUT_GBIF}, {OUT_QC}")
clean.head()

In [ ]:
# ---- Optional: download the outputs straight to your machine ----------
try:
    from google.colab import files
    for f in [OUT_CLEAN, OUT_QC, OUT_OBIS, OUT_GBIF]:
        if os.path.exists(f):
            files.download(f)
except ImportError:
    print("Not running in Colab — files are in the working directory.")

## 11. Plan A reality check — how empty is Mafia Island in open data?

This cell exists to be shown, not to be worked around. It demonstrates empirically what the
dataset review concluded: the global open aggregators hold almost nothing usable inside the
Mafia Island polygon, and what they do hold is obscured to roughly the size of the study area.

That finding is what justifies both the Sharkbook export and the MMF request — it is evidence,
not an excuse.

In [ ]:
# ---- Mafia Island subset ----------------------------------------------
m = clean[(clean["lon"].between(MAFIA_BBOX["lon_min"], MAFIA_BBOX["lon_max"])) &
          (clean["lat"].between(MAFIA_BBOX["lat_min"], MAFIA_BBOX["lat_max"]))].copy()

print(f"Records inside the Mafia Island polygon: {len(m)}")
if len(m):
    print(f"  obscured coordinates: {int(m['is_obscured'].sum())} of {len(m)}")
    print(f"  median coordinate uncertainty: {m['uncertainty_m'].median()} m")
    print(f"  date range: {m['datetime'].min()} -> {m['datetime'].max()}")
    print(f"  sources: {m['source'].value_counts().to_dict()}")
    print()
    print("Bay width is roughly 20 km. Compare that with the uncertainty above before")
    print("treating any of these points as a location within the bay.")
m

## 12. Plan A seed — Cagua et al. (2015), Dryad

**Download link:** https://datadryad.org/dataset/doi:10.5061/dryad.g6c5q
(DOI `10.5061/dryad.g6c5q` — open, no registration, 7 files, ~116 KB)

Cagua, E.F., Cochran, J.E.M., Rohner, C.A., Prebble, C.E.M., Sinclair-Taylor, T.H., Pierce,
S.J., Berumen, M.L. (2015). *Acoustic telemetry reveals cryptic residency of whale sharks.*
Biology Letters.

**Why this deposit matters more than its size suggests.** It contains `searching-hours.csv` —
the hours spent looking. That is the denominator. Every other source in this project records
only where someone looked *and found something*; without effort, a model cannot tell "no sharks
here" from "nobody went here", and the whole presence/absence framing collapses into
presence-only guesswork.

Download the zip, unzip it, and upload `searching-hours.csv` (plus the sightings file) with the
cell below.

In [ ]:
# ---- Upload the Cagua deposit files -----------------------------------
try:
    from google.colab import files
    print("Select the unzipped Cagua et al. 2015 CSV files (searching-hours.csv and the")
    print("sightings/detections files):")
    uploaded = files.upload()
    for name in uploaded:
        print(f"  uploaded: {name}")
except ImportError:
    print("Not in Colab — place the Cagua CSV files in the working directory manually.")

In [ ]:
# ---- Inspect whatever was uploaded -------------------------------------
import glob

cagua_files = sorted([f for f in glob.glob("*.csv")
                      if f not in {OUT_CLEAN, OUT_OBIS, OUT_GBIF}])

for f in cagua_files:
    try:
        d = pd.read_csv(f)
        print(f"\n=== {f} ===")
        print(f"shape: {d.shape}")
        print(f"columns: {list(d.columns)}")
        display(d.head(3))
    except Exception as e:
        print(f"{f}: could not read ({e})")

## 13. What to do next — in this order

Getting the order wrong is the most likely way this project produces an impressive number that
means nothing.

**1. Build the seasonal climatology baseline first.** Historical presence rate by month. Rohner
et al. (2020) already established that whale shark presence at Mafia Island is seasonally
predictable, so *knowing the month* is a strong predictor on its own. If the model does not beat
month-of-year, there is no operational contribution — and that must be reported plainly rather
than buried. Build this in week 6, not week 12.

**2. Handle background sampling before modelling anything.** These are presence-only data with
no effort field. Generate background points from a plausible accessible marine domain using a
**target-group background** — sample from where *other* marine megafauna recording happened, so
the model learns "whale shark vs. other sightings" rather than "surveyed vs. unsurveyed ocean".

**3. Extract environmental covariates** onto the presence points *and* the background points:
SST and chlorophyll-a (MODIS/VIIRS or Copernicus), bathymetry (GEBCO), tide state, lunar phase
(already computed here). Be honest in writing that chlorophyll-a is a coarse proxy — the actual
prey at Mafia is sergestid shrimp (Rohner et al., *J. Plankton Res.*), which is not remotely
sensed.

**4. Validate spatially and temporally, never with random k-fold.** Occurrence records are
spatially clustered and seasonally structured; random folds leak between train and test and
produce beautiful, meaningless AUC values. Use spatial block CV and a temporal split (train on
earlier years, test on later).

**5. Model families are fixed:** logistic regression as the interpretable baseline, gradient
boosting / Random Forest as the challenger, MaxEnt-style presence-background for the global
track. No deep learning — the effective sample size does not justify it.

**6. Metrics:** PR-AUC as primary (the classes are heavily imbalanced, so accuracy is
meaningless), ROC-AUC for comparability with the species-distribution literature, and a
calibration curve — which matters most, because an operator acting on "70%" needs it to mean 70%.

---

### Not a modelling step, but the real critical path

Use of the Sharkbook Tanzania records in any publication or product requires the **written
consent of the original data provider**. For Mafia Island that is MMF, which is about 30% of
those records. The request routed through WATONET is therefore not just a route to more data —
it is the consent gate for the data already in hand.
